# Lifecycle & Wrap Hooks

This notebook breaks down each middleware hook into its own dedicated section:
1. Agent Lifecycle Hooks (before_agent, after_agent)
2. Model Lifecycle Hooks (before_model, after_model)
3. Tool Wrap Hooks (wrap_tool_call)
4. Full Combined Execution Pipeline

### 1. Agent Lifecycle Hooks (before_agent & after_agent)

- **before_agent (on_chain_start)**: Fires once when the overall agent workflow starts.
- **after_agent (on_chain_end)**: Fires once when the agent finishes its work.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_core.callbacks import BaseCallbackHandler
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(find_dotenv())

class AgentLifecycleLogger(BaseCallbackHandler):
    def on_chain_start(self, serialized, inputs, **kwargs):
        print("🚀 [before_agent] Starting agent execution session...")
    
    def on_chain_end(self, outputs, **kwargs):
        print("🏁 [after_agent] Finished agent execution session!")

agent_logger = AgentLifecycleLogger()
print("Agent Lifecycle Logger created!")


Agent Lifecycle Logger created!


### 2. Model Lifecycle Hooks (before_model & after_model)

- **before_model (on_llm_start)**: Fires right before sending a prompt to the LLM API.
- **after_model (on_llm_end)**: Fires right after receiving a response from the LLM.

In [1]:
class ModelLifecycleLogger(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print("🔍 [before_model] Preparing prompt... Sending request to Gemini 3.6 Flash.")
    
    def on_llm_end(self, response, **kwargs):
        print("📝 [after_model] Response received successfully from Gemini 3.6 Flash.")

model_logger = ModelLifecycleLogger()
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", callbacks=[model_logger])

print("=== Testing Model Lifecycle Hooks ===")
response = llm.invoke("Explain gravity in one short sentence.")
print(f"Result: {response.content}")


=== Testing Model Lifecycle Hooks ===
🔍 [before_model] Preparing prompt... Sending request to Gemini 3.6 Flash.
📝 [after_model] Response received successfully from Gemini 3.6 Flash.
Result: [{'type': 'text', 'text': 'Gravity is the invisible force that pulls objects with mass toward one another.', 'extras': {'signature': 'EqUNCqINARFNMg95WcyYN4iDqOa81Vzf7jJhxXiap/Mzd1v+oYGsIQ85es2WU8WIPS2V0bNeBIE4jnjqow9EhbFokoZ2ns3cQKAaKXSNrN3e/DgU2NMGTvgvEnFB9jojk4+PuMjToaTJvOuthwQO5ZuIOtDuJ9ee/r8E5FltEqLGTDX4vM7hQJlrFCUrhpE4GniMlcEXG2uRAAzSCJou9jrYD1EeYoi+Q/04HbiZiDKcNz+qHzDphrmlcr3L5X3IRLP5aeDDJnTiwt32Fq0opfcpSSlIetSE9BmHksZDA+4VLOMLEr4MuMHT/x32ygheLMyHMYmTrHxTEHH1hJnP8D9i6K+htKzBGuuhCVCBX8Fwz9I+pbwLoxxZhINilZGM6Gkxn6LXc40OyYhCy1gylvHkVLxXGX54SQXkIbmdeoIRwVIbV0HXEBpwlSZ8ewzBoVfm0QIFsgGxI334DGREZoogDjFebIXS9mUZhxdHPrD5E/Vn9loLd6rnTHmd3F1BFp4snYiNrroomT4i4T0K03vP9+0rc9yv3tpIaiZ5DD0XVHeqnHOTM3FVn/fCTkDvZeTX3B/ALMFBH8Nd8W0cbXRyyqz3MxmSnbP7w5GtSvgfiWpwONARJ0lX+fODtoLPObvxspkFNo1JoEmavuTbafleer1wXZlpmbJB

### 3. Tool Wrap Hooks (wrap_tool_call)

- **on_tool_start**: Wraps around the beginning of tool execution (intercepts tool input arguments).
- **on_tool_end**: Wraps around the end of tool execution (intercepts tool return value).

In [1]:
from langchain_core.tools import tool

class ToolWrapLogger(BaseCallbackHandler):
    def on_tool_start(self, serialized, input_str, **kwargs):
        tool_name = serialized.get("name", "tool")
        print(f"🛠️ [wrap_tool_call: START] Intercepted tool '{tool_name}' with input: {input_str}")
    
    def on_tool_end(self, output, **kwargs):
        print(f"✅ [wrap_tool_call: END] Tool output: {output}")

@tool
def calculate_square(num: float) -> float:
    """Calculate and return the square of a number."""
    return num * num

tool_logger = ToolWrapLogger()
print("Tool Wrap Logger created!")


Tool Wrap Logger created!


### 4. Complete Combined Execution Pipeline

We attach all hooks together to see the full, clean execution trace of an agent tool call!

In [1]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

combined_handlers = [agent_logger, model_logger, tool_logger]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a math assistant equipped with a square tool."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, [calculate_square], prompt)
agent_executor = AgentExecutor(agent=agent, tools=[calculate_square], callbacks=combined_handlers)

print("=== Running Agent Pipeline with All Hooks Combined ===\n")
result = agent_executor.invoke({"input": "What is the square of 12?"})
print(f"\nFinal Result: {result['output']}")


=== Running Agent Pipeline with All Hooks Combined ===

🚀 [before_agent] Starting agent execution session...
🔍 [before_model] Preparing prompt... Sending request to Gemini 3.6 Flash.
📝 [after_model] Response received successfully from Gemini 3.6 Flash.
🔍 [before_model] Preparing prompt... Sending request to Gemini 3.6 Flash.
📝 [after_model] Response received successfully from Gemini 3.6 Flash.
🏁 [after_agent] Finished agent execution session!

Final Result: [{'type': 'text', 'text': 'The square of 12 is 144.', 'index': 0, 'extras': {'signature': 'EnsKeQERTTIPgOBLwt2tSrDm8P8YmOYJ47cIiasi18zAbljjUe9PAusgLGUAoDw2NjlonFOqbmENOkFNs5ZMt67etrJFExskyJyrMKFe7K8f+g6SFrFdYFMpKce9rDml5vTelhOpPIuZ9STNhzO364qpq86VvkOdroNwo6Q='}}]
